In [ ]:
# Copyright (c) TorchGeo Contributors. All rights reserved.
# Licensed under the MIT License.

# Evaluating SSL Checkpoints

_Written by: Caleb Robinson_

This tutorial shows how to evaluate a self-supervised learning (SSL) checkpoint. We train SimCLR on EuroSAT100 and use a k-nearest-neighbors (kNN) classifier to predict image classes from the encoder's features, without changing its weights.

The two-epoch run is a short demonstration. For results on the full EuroSAT dataset, see the [SSL benchmark](../user/ssl_benchmark.rst).

## Setup

To do a torchgeo tutorial, we need torchgeo.

In [ ]:
!uv pip install torchgeo scikit-learn

## Imports

In [ ]:
import os
import tempfile

import lightning.pytorch as pl
import numpy as np
import timm
import torch
import torch.nn.functional as F
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from torch import Tensor, nn

from torchgeo.datamodules import EuroSAT100DataModule
from torchgeo.tasks import SimCLR

## Data

First we'll create a `EuroSAT100DataModule` object which is simply a wrapper around the [EuroSAT100](https://docs.torchgeo.org/en/latest/api/datasets/eurosat.html) dataset (an 100-image subset of EuroSAT).

In [ ]:
root = os.path.join(tempfile.gettempdir(), 'tasks')
datamodule = EuroSAT100DataModule(root=root, batch_size=8, num_workers=0, download=True)
# downloads the datasets
datamodule.setup('fit')
datamodule.setup('validate')

## Pretraining a task

We train SimCLR for an epoch with `in_channels=13` for EuroSAT's 13 bands. Setting `size=64` keeps the images at their default (small) size so that the tutorial can be run quickly on a CPU.

The [SimCLR ResNet-50 configuration](https://github.com/torchgeo/torchgeo/blob/main/tests/configs/ssl_benchmarking/simclr_resnet50.yaml) in the [SSL benchmark](../user/ssl_benchmark.rst) trains for 60 epochs with `224 x 224` crops.

In [ ]:
pl.seed_everything(0)

task = SimCLR(
    model='resnet18', in_channels=13, version=2, lr=0.5, size=64, memory_bank_size=0
)

trainer = pl.Trainer(
    accelerator='auto',
    devices=1,
    max_epochs=2,
    # SimCLR does not run validation.
    limit_val_batches=0,
    num_sanity_val_steps=0,
    enable_checkpointing=False,
    enable_progress_bar=False,
    logger=False,
)
trainer.fit(model=task, datamodule=datamodule)

checkpoint_path = os.path.join(tempfile.gettempdir(), 'simclr_eurosat100.ckpt')
trainer.save_checkpoint(checkpoint_path)

## Loading the encoder

We create a [timm](https://huggingface.co/docs/timm/index) model using the settings saved in the checkpoint and load the encoder's weights. The weight names use these prefixes:

| Task | Weight name prefix |
| --- | --- |
| `SimCLR` | `backbone.` |
| `MoCo` | `backbone.` |
| `BYOL` | `model.backbone.model.` |

In [ ]:
def load_backbone(path: str) -> nn.Module:
    """Load the encoder weights from an SSL checkpoint."""
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    hparams = checkpoint['hyper_parameters']
    state_dict = checkpoint['state_dict']

    prefix = None
    for prefix in ('model.backbone.model.', 'backbone.'):
        if any(key.startswith(prefix) for key in state_dict):
            break

    weights = {
        key[len(prefix) :]: value
        for key, value in state_dict.items()
        if key.startswith(prefix)
    }

    backbone = timm.create_model(
        hparams['model'], in_chans=hparams['in_channels'], num_classes=0
    )
    backbone.load_state_dict(weights, strict=True)
    return backbone.eval()


backbone = load_backbone(checkpoint_path)

## Extracting features

During training, Lightning applies the datamodule's normalization in `on_after_batch_transfer`. Here, we call `datamodule.aug` ourselves, then resize the images to the training resolution without random augmentations.

We use a separate dataloader that includes every image as the training dataloader from the datamodule drops the last batch if it is incomplete.

`forward_head(forward_features(x), pre_logits=True)` returns features from before the classification layer for both convolutional and transformer encoders.

In [ ]:
@torch.no_grad()
def extract_features(
    backbone: nn.Module, dataset: torch.utils.data.Dataset, size: int = 64
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features and labels for each image."""
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=False)
    features, labels = [], []
    for batch in dataloader:
        x: Tensor = batch['image']
        x = datamodule.aug({'image': x})['image']
        x = F.interpolate(x, size=(size, size), mode='bilinear', align_corners=False)
        z = backbone.forward_head(backbone.forward_features(x), pre_logits=True)
        features.append(z.flatten(1))
        labels.append(batch['label'])

    return torch.cat(features).numpy(), torch.cat(labels).numpy()


x_train, y_train = extract_features(backbone, datamodule.train_dataset)
x_val, y_val = extract_features(backbone, datamodule.val_dataset)

print(f'train features {x_train.shape}, val features {x_val.shape}')

## Fitting the kNN classifier

The classifier predicts each validation image's class from its five nearest training examples.

We evaluate both the original features and features standardized with `StandardScaler`. Scaling changes the distances used by kNN, so we report both scores and keep the better one.

In [ ]:
def knn_score(
    x_train: np.ndarray, y_train: np.ndarray, x_eval: np.ndarray, y_eval: np.ndarray
) -> dict[str, float]:
    """Evaluate raw and standardized features with kNN."""
    scores = {}
    for name in ('raw', 'standardized'):
        if name == 'raw':
            a, b = x_train, x_eval
        else:
            scaler = StandardScaler()
            a, b = scaler.fit_transform(x_train), scaler.transform(x_eval)
        probe = KNeighborsClassifier(n_neighbors=5).fit(a, y_train)
        scores[name] = float(probe.score(b, y_eval))
    scores['best'] = max(scores['raw'], scores['standardized'])
    return scores


scores = knn_score(x_train, y_train, x_val, y_val)
for name, value in scores.items():
    print(f'{name:>13}: {value:.4f}')

## Comparing with an untrained model

We repeat the evaluation with the same encoder architecture and random weights to see whether pretraining improved accuracy.

With only 20 validation images, one correct prediction changes accuracy by 0.05. Use the full [SSL benchmark](../user/ssl_benchmark.rst) for a larger comparison.

In [ ]:
random_backbone = timm.create_model('resnet18', in_chans=13, num_classes=0).eval()
x_train_random, _ = extract_features(random_backbone, datamodule.train_dataset)
x_val_random, _ = extract_features(random_backbone, datamodule.val_dataset)

random_scores = knn_score(x_train_random, y_train, x_val_random, y_val)
print(f'   pretrained: {scores["best"]:.4f}')
print(f'random init.: {random_scores["best"]:.4f}')

## Checking for representation collapse

An encoder has collapsed if it produces nearly identical features for different images. We normalize each feature vector to length one and check:

- Standard deviation across images: values near zero mean little variation.
- Mean pairwise cosine similarity: values near one mean the feature vectors point in nearly the same direction.

Report these checks alongside accuracy. In the benchmark, one MoCo run reached a standard deviation of 0.0005 and scored below the untrained model while its loss barely changed.

In [ ]:
def collapse_diagnostics(features: np.ndarray) -> dict[str, float]:
    """Check how similar the features are across images."""
    normalized = F.normalize(torch.from_numpy(features).float(), dim=1)
    similarity = normalized @ normalized.T
    n = len(normalized)
    # Exclude the diagonal, which is always 1.
    off_diagonal = (similarity.sum() - n) / (n * (n - 1))
    return {
        'embedding_std': float(normalized.std(dim=0).mean()),
        'mean_pairwise_cosine': float(off_diagonal),
    }


for name, features in (('pretrained', x_train), ('random init.', x_train_random)):
    stats = collapse_diagnostics(features)
    print(
        f'{name:>12}: std={stats["embedding_std"]:.4f} '
        f'cosine={stats["mean_pairwise_cosine"]:.4f}'
    )

## Running the full benchmark

Use the configurations in [tests/configs/ssl_benchmarking](https://github.com/torchgeo/torchgeo/tree/main/tests/configs/ssl_benchmarking) to reproduce the full EuroSAT runs for SimCLR, MoCo, and BYOL. The [SSL benchmark](../user/ssl_benchmark.rst) describes the settings and results.

When trying new settings, compare learning rates on the validation split, then evaluate the selected model once on the test split.